# Bimodal Figure 1 from the Oct 5 xseed data (moment-mode $\tilde\Theta$)

Reads `lam_selection_criterion/results/bimodal_xseed/beta_*/run_*_s*.pt`
(10 datasets $\times$ 3 SDE seeds per $\beta$, 3 $\beta$ values; shared moment-mode SDE, uniform time weights, ridge $10^{-4}$).
Compared to `bimodal_experiment/kl_bimodal.ipynb` (`KL_bimodal_paper_1.png`, Sep 24) the method is the one the paper decided on (2026-10-05):
**$\tilde\Theta$ of eqs. 5–6 of the draft, uniform weights, $\lambda$ by cross-seed CV scored on $t\ge 0.99$**.
The old figure used the legacy $\tau$-based REG of `bimodal_experiment/sde_routines_scalar_reg.py`.

**Caveats** (for the caption):
- 30 points per $\beta$ instead of the Sep 24 figure's 100; 10 datasets $\times$ 3 seeds. Seeds share $x_1$; the variance seen here undercounts the dataset variance.
- 3 $\beta$ values (0.1, 0.9, 1.9) instead of 10; the 10-$\beta$ grid needs the full xseed sweep run on Jean Zay (`BETAS="0.1 0.3 ... 1.9" bash lam_selection_criterion/launch_bimodal.sh`).
- $\lambda$ is picked per dataset (shared by its 3 seeds); the diagnostic cell prints the $\lambda$ histogram per $\beta$.

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.integrate import simpson

# repo root; the notebook finds itself whether on the cluster or locally
ROOT = Path.cwd()
while not (ROOT / '.git').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
RESULTS = Path(os.environ.get('BM_XSEED_DIR', ROOT / 'lam_selection_criterion' / 'results' / 'bimodal_xseed'))
RULE = os.environ.get('BM_RULE', 'uniform_xseed_end0.99')  # the paper's chosen rule (see lam_selection_criterion/README.md)
SCHED = 'uniform'  # the paper's chosen weighting
X_RANGE, N_GRID = (-10.0, 10.0), 8001
print('RESULTS :', RESULTS)
print('rule    :', RULE)

In [ ]:
# ── load every per-replica result file, group by beta ─────────────────────
def _theta_key(lam):
    return f'{SCHED}|{lam:g}'                                      # the format used by bimodal_lamtest.py save

def load_beta_dir(d):
    files = sorted(d.glob('run_*_s*.pt'))
    runs = [torch.load(f, map_location='cpu', weights_only=False) for f in files]
    assert runs, f'no run_*_s*.pt in {d}'
    th_star = runs[0]['theta_star'].double().numpy()
    for r in runs[1:]:                                             # theta* depends only on beta
        assert np.allclose(r['theta_star'].double().numpy(), th_star)
    th_mgd, th_reg, lams = [], [], []
    for r in runs:
        lam = float(r['rules'][RULE]['lam'])
        th_mgd.append(r['theta_end']['theta_t'].double().numpy())  # the raw per-step theta at t=1
        th_reg.append(r['theta_end'][_theta_key(lam)].double().numpy())
        lams.append(lam)
    return dict(beta=float(runs[0]['beta']), theta_star=th_star,
                mgd=np.array(th_mgd), reg=np.array(th_reg),
                lams=np.array(lams), n_runs=len(runs),
                cr_bound=float(runs[0]['cr_bound']), kl_floor=float(runs[0]['kl_floor']))

per_beta = []
for d in sorted(RESULTS.glob('beta_*')):
    if d.is_dir() and any(d.glob('run_*_s*.pt')):
        per_beta.append(load_beta_dir(d))
assert per_beta, f'no beta_*/run_*_s*.pt under {RESULTS}'
betas = np.array([r['beta'] for r in per_beta])
print(f'{len(per_beta)} beta values loaded: {betas}')
for r in per_beta:
    uniq, cnt = np.unique(r['lams'], return_counts=True)
    hist = ', '.join(f'{l:g}:{c}' for l, c in zip(uniq, cnt))
    print(f"  beta {r['beta']:4.2f}: {r['n_runs']} runs | lam ({RULE}): {hist}")

In [ ]:
# ── KL on the GRID; same convention as bimodal_experiment/kl_bimodal.ipynb ───
# log p_theta(x) = theta . phi(x) + const, phi_a = x^a / a, a = 1..4
GRID = np.linspace(*X_RANGE, N_GRID)
PHI = np.stack([GRID, GRID**2 / 2, GRID**3 / 3, GRID**4 / 4], axis=1)

def log_density(thetas):
    thetas = np.atleast_2d(np.asarray(thetas, dtype=float))
    lv = thetas @ PHI.T
    m = lv.max(axis=1, keepdims=True)
    with np.errstate(over='ignore'):
        lv = lv - (m + np.log(simpson(np.exp(lv - m), x=GRID, axis=1))[:, None])
    lv[thetas[:, 3] >= 0] = np.nan                                 # not normalisable on R
    return lv

def kl_rows(log_p_ref, thetas, chunk=200):
    p = np.exp(log_p_ref)
    out = []
    for i in range(0, len(thetas), chunk):
        with np.errstate(invalid='ignore'):
            f = p * (log_p_ref - log_density(thetas[i:i + chunk]))
        out.append(simpson(np.where(p > 0, f, 0.0), x=GRID, axis=1))
    return np.concatenate(out)

# sanity: KL(p_true || p_theta*) should be ~0 (basis convention check)
for r in per_beta:
    log_p_true = log_density(r['theta_star'])[0]
    print(f"beta {r['beta']:4.2f}: KL(p_theta* || p_theta*) = {kl_rows(log_p_true, r['theta_star'][None])[0]: .2e}")

In [ ]:
# ── per-beta KL summary: variance (KL vs mean of estimator) and bias (KL vs theta*) ──
rows = []
for r in per_beta:
    mgd, reg = r['mgd'], r['reg']
    # paper's figure metric: KL(p_mean || p_hat), one value per run
    mgd_mean = kl_rows(log_density(mgd.mean(0))[0], mgd)
    reg_mean = kl_rows(log_density(reg.mean(0))[0], reg)
    # vs the exact theta* (bias + variance, the KL the slide text usually cites)
    log_p_true = log_density(r['theta_star'])[0]
    mgd_true = kl_rows(log_p_true, mgd)
    reg_true = kl_rows(log_p_true, reg)
    rows.append(dict(beta=r['beta'], K=len(mgd),
                     mgd_mean=mgd_mean, reg_mean=reg_mean,
                     mgd_true=mgd_true, reg_true=reg_true,
                     cr_bound=r['cr_bound'], kl_floor=r['kl_floor']))

print(f"{'beta':>5} {'K':>4} | {'KL vs mean MGD':>14} {'REG':>10} {'gain':>6} | "
      f"{'KL vs truth MGD':>15} {'REG':>10} | {'CR':>8} {'floor':>8} | NaN")
for r in rows:
    nan = sum(np.isnan(r[k]).sum() for k in ('mgd_mean', 'reg_mean', 'mgd_true', 'reg_true'))
    mm, rm = np.nanmean(r['mgd_mean']), np.nanmean(r['reg_mean'])
    gain = mm / rm if rm > 0 else np.nan
    print(f"{r['beta']:5.2f} {r['K']:4d} | {mm:14.3e} {rm:10.3e} {gain:6.0f} | "
          f"{np.nanmean(r['mgd_true']):15.3e} {np.nanmean(r['reg_true']):10.3e} | "
          f"{r['cr_bound']:8.1e} {r['kl_floor']:8.1e} | {nan}")

In [ ]:
# ── Figure 1 (same style as bimodal_experiment/kl_bimodal.ipynb) ──────────
C_MGD, C_REG, INK, MUTED = '#2a78d6', '#eb6834', '#0b0b0b', '#52514e'
cr = rows[0]['cr_bound']

with plt.rc_context({'font.size': 16, 'axes.labelsize': 22, 'xtick.labelsize': 17, 'ytick.labelsize': 17,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': MUTED, 'xtick.color': MUTED, 'ytick.color': MUTED,
                     'axes.labelcolor': INK}):
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    for key, col, lab, mk in [('mgd_mean', C_MGD, 'MGD', 'o'), ('reg_mean', C_REG, 'Regularized MGD', 's')]:
        vals = [r[key][np.isfinite(r[key])] for r in rows]
        lo, hi = np.array([np.percentile(v, [25, 75]) for v in vals]).T
        ax.fill_between(betas, lo, hi, color=col, alpha=0.15, lw=0)        # 25-75 % spread over the runs
        ax.plot(betas, [v.mean() for v in vals], '-' + mk, color=col, lw=2.2, ms=9,
                mfc='white', mec=col, mew=2, label=lab, zorder=3)
    ax.axhline(cr, color=MUTED, ls='--', lw=1.8, label='Cram\u00e9r\u2013Rao bound')
    ax.axhspan(cr / 100, cr, color='0.93', lw=0, zorder=0)                 # unreachable region
    ax.set_yscale('log')
    ax.set_xlabel(r'$|\theta|$')                                             # x values are beta, label as in the paper
    ax.set_ylabel(r'$\mathbb{E}\,[D_{\rm KL}(p_{\bar\theta}\,\Vert\, p_{\hat\theta})]$', fontsize=19)
    ax.set_ylim(cr / 5, None)
    ax.tick_params(width=1.2, length=5)
    ax.grid(axis='y', which='major', color='0.85', lw=0.8)
    ax.set_axisbelow(True)
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False,
              handlelength=1.8, columnspacing=1.2, handletextpad=0.5, fontsize=15)
    out = RESULTS / 'KL_bimodal_xseed_paper.png'
    plt.tight_layout()
    plt.savefig(out, dpi=200, bbox_inches='tight')
    plt.show()
print('wrote', out)

In [ ]:
# ── diagnostic: KL vs the exact theta* (bias + variance; what the slide text cites) ───
with plt.rc_context({'font.size': 16, 'axes.labelsize': 22, 'xtick.labelsize': 17, 'ytick.labelsize': 17,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': MUTED, 'xtick.color': MUTED, 'ytick.color': MUTED,
                     'axes.labelcolor': INK}):
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    for key, col, lab, mk in [('mgd_true', C_MGD, 'MGD', 'o'), ('reg_true', C_REG, 'Regularized MGD', 's')]:
        vals = [r[key][np.isfinite(r[key])] for r in rows]
        lo, hi = np.array([np.percentile(v, [25, 75]) for v in vals]).T
        ax.fill_between(betas, lo, hi, color=col, alpha=0.15, lw=0)
        ax.plot(betas, [v.mean() for v in vals], '-' + mk, color=col, lw=2.2, ms=9,
                mfc='white', mec=col, mew=2, label=lab, zorder=3)
    ax.plot(betas, [r['kl_floor'] for r in rows], ':', color=MUTED, lw=1.8, label='max-likelihood floor')
    ax.set_yscale('log')
    ax.set_xlabel(r'$|\theta|$')
    ax.set_ylabel(r'$\mathbb{E}\,[D_{\rm KL}(p_{\theta^*}\,\Vert\, p_{\hat\theta})]$', fontsize=19)
    ax.tick_params(width=1.2, length=5)
    ax.grid(axis='y', which='major', color='0.85', lw=0.8)
    ax.set_axisbelow(True)
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False,
              handlelength=1.8, columnspacing=1.2, handletextpad=0.5, fontsize=15)
    plt.tight_layout()
    plt.savefig(RESULTS / 'KL_bimodal_xseed_vs_truth.png', dpi=200, bbox_inches='tight')
    plt.show()